In [ ]:
!pip install pyannote.audio==4.0.0
!pip install torch==2.8.0 --index-url https://download.pytorch.org/whl/cu126
!pip install torchaudio==2.8.0 torchvision==0.23.0 --index-url https://download.pytorch.org/whl/cu126
!pip install torchcodec==0.7
!pip install -q 'datasets>=2.14' soundfile

print("\n✅ Cài đặt xong!")
print("🔄 RESTART RUNTIME NGAY BÂY GIỜ: Runtime → Restart session")

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import os
import torch
import numpy as np
from google.colab import drive

# ============================================================
# Mount Google Drive để lưu checkpoints liên tục
# ============================================================
print("Mounting Google Drive...")
drive.mount('/content/drive')

# ============================================================
# PyTorch patch
# ============================================================
_orig_load = torch.load
def _patched_load(f, *args, **kwargs):
    kwargs['weights_only'] = False
    return _orig_load(f, *args, **kwargs)
torch.load = _patched_load
print("✓ PyTorch load function patched")

# ============================================================
# NumPy patch
# ============================================================
if not hasattr(np, 'NaN'):
    np.NaN = np.nan
    print("✓ Added np.NaN")
if not hasattr(np, 'Inf'):
    np.Inf = np.inf
    print("✓ Added np.Inf")

# ============================================================
# GPU check
# ============================================================
assert torch.cuda.is_available(), "❌ Không có GPU! Vào Runtime → Change runtime type → T4 GPU"
device = torch.device('cuda')
print(f'\n✓ PyTorch {torch.__version__}')
print(f'✓ GPU: {torch.cuda.get_device_name(0)}')
print(f'✓ Device: {device}')

In [ ]:
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN')
    print("✅ Đọc token từ Colab Secrets")
except Exception:
    # Nếu chưa cài Secrets, bạn điền trực tiếp token của bạn vào đây
    HF_TOKEN = "YOUR_HF_TOKEN"
    print("⚠️ Dùng token điền tay")

assert HF_TOKEN and not HF_TOKEN.startswith("hf_PASTE"), "❌ Chưa điền HuggingFace token!"
print(f"✅ Token hợp lệ: {HF_TOKEN[:8]}...")

In [ ]:
import os
import soundfile as sf
from datasets import load_dataset

# ============================================================
# ⚙️ CẤU HÌNH CONTINUAL LEARNING
# ============================================================
CURRENT_PART = 3
SAMPLES_PER_PART_TRAIN = 70
SAMPLES_PER_PART_TEST = 5

BASE_DRIVE_DIR = '/content/drive/MyDrive/VoxConverse_Continual_Learning'
CHECKPOINT_DIR       = str(os.path.join(BASE_DRIVE_DIR, f'checkpoints_part_{CURRENT_PART}'))
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

START_IDX_TRAIN = (CURRENT_PART - 1) * SAMPLES_PER_PART_TRAIN
END_IDX_TRAIN = START_IDX_TRAIN + SAMPLES_PER_PART_TRAIN

START_IDX_TEST = (CURRENT_PART - 1) * SAMPLES_PER_PART_TEST
END_IDX_TEST = START_IDX_TEST + SAMPLES_PER_PART_TEST

print(f"========== CONFIGURING: PART {CURRENT_PART} ==========")
print(f"👉 Sẽ lấy các mẫu dữ liệu dùng để train và validation: {START_IDX_TRAIN} đến {END_IDX_TRAIN} (Tổng: {SAMPLES_PER_PART_TRAIN} mẫu)")

print(f"👉 Sẽ lấy các mẫu dữ liệu dùng để test: {START_IDX_TEST} đến {END_IDX_TEST} (Tổng: {SAMPLES_PER_PART_TEST} mẫu)")

# 1 BIẾN GỐC DUY NHẤT: Thư mục dự án tổng trên Google Drive
WORK_DIR        = '/content/voxconverse_mini'
TRAIN_AUDIO_DIR = os.path.join(WORK_DIR, 'audio', 'train')
TEST_AUDIO_DIR  = os.path.join(WORK_DIR, 'audio', 'test')
RTTM_DIR        = os.path.join(WORK_DIR, 'rttm')
UEM_DIR         = os.path.join(WORK_DIR, 'uem')
LST_DIR         = os.path.join(WORK_DIR, 'lst')
OUTPUT_DIR      = '/content/output'

for d in [TRAIN_AUDIO_DIR, TEST_AUDIO_DIR, RTTM_DIR, UEM_DIR, LST_DIR, OUTPUT_DIR]:
    os.makedirs(d, exist_ok=True)

print("✓ Thư mục đã tạo")

# ============================================================
# TẢI DATASET (STREAMING) THEO PHẠM VI IDX ĐÃ TÍNH TOÁN
# ============================================================
print(f"\n📥 Đang kết nối tới HuggingFace VoxConverse (Streaming)...")
ds_dev = load_dataset("diarizers-community/voxconverse", split="dev", token=HF_TOKEN, streaming=True)
ds_test = load_dataset("diarizers-community/voxconverse", split="test", token=HF_TOKEN, streaming=True)

mini_train = list(ds_dev.skip(START_IDX_TRAIN).take(SAMPLES_PER_PART_TRAIN))
mini_test  = list(ds_test.skip(START_IDX_TEST).take(SAMPLES_PER_PART_TEST))
print(f"✅ Đã lấy thành công {len(mini_train)} mẫu dữ liệu từ dev split.")
print(f"✅ Đã lấy thành công {len(mini_test)} mẫu dữ liệu từ dev split.")

# ============================================================
# HELPER FUNCTIONS & TRÍCH XUẤT NHÃN GỐC
# ============================================================
def make_rttm(uri, starts, ends, speakers):
    lines = []
    for s, e, spk in zip(starts, ends, speakers):
        dur = round(float(e) - float(s), 3)
        if dur <= 0:
            continue
        lines.append(f"SPEAKER {uri} 1 {float(s):.3f} {dur:.3f} <NA> <NA> {spk} <NA> <NA>")
    return "\n".join(lines) + "\n" if lines else ""

def make_uem(uri, duration):
    return f"{uri} 1 0.000 {duration:.3f}\n"

# ============================================================
# Xử lý train splits: lưu audio .wav + tạo RTTM + UEM
# ============================================================
def process_split(dataset, audio_dir, split_name):
    uris = []
    for i, sample in enumerate(dataset):
        # Đặt tên file theo split + index (train_000, test_000, ...)
        uri = f"{split_name}_{i:03d}"
        uris.append(uri)

        # Lưu audio
        audio  = sample['audio']
        arr    = np.array(audio['array'], dtype=np.float32)
        sr     = audio['sampling_rate']
        wav_path = os.path.join(audio_dir, f"{uri}.wav")
        sf.write(wav_path, arr, sr)

        duration = len(arr) / sr

        # RTTM
        rttm_content = make_rttm(
            uri,
            sample['timestamps_start'],
            sample['timestamps_end'],
            sample['speakers'],
        )
        with open(os.path.join(RTTM_DIR, f"{uri}.rttm"), 'w') as f:
            f.write(rttm_content)

        # UEM
        with open(os.path.join(UEM_DIR, f"{uri}.uem"), 'w') as f:
            f.write(make_uem(uri, duration))

        n_spk = len(set(sample['speakers']))
        print(f"  [{i+1}/{len(dataset)}] {uri}.wav  dur={duration:.1f}s  speakers={n_spk}")

    return uris


print("⚙️  Xử lý train files...")
train_uris = process_split(mini_train, TRAIN_AUDIO_DIR, 'train')

print("\n⚙️  Xử lý test files...")
test_uris = process_split(mini_test, TEST_AUDIO_DIR, 'test')

print("\n✅ Đã lưu audio + RTTM + UEM")

In [ ]:
dev_uris = train_uris[-5:]
train_uri_list = train_uris[:-5]

def write_lst(path, uris):
    with open(path, 'w') as f:
        f.write("\n".join(uris) + "\n")

write_lst(os.path.join(LST_DIR, 'train.lst'), train_uri_list)
write_lst(os.path.join(LST_DIR, 'development.lst'), dev_uris)
write_lst(os.path.join(LST_DIR, 'test.lst'), test_uris)

print("✅ LST files:")
print(f"   train.lst       : {train_uri_list}")
print(f"   development.lst : {dev_uris}")
print(f"   test.lst        : {test_uris}")

In [ ]:
# ============================================================
# Tạo database.yml
# ============================================================
DB_YML_PATH = os.path.join(WORK_DIR, 'database.yml')

# Merge tất cả RTTM thành 1 file
all_rttm_path = os.path.join(RTTM_DIR, 'all.rttm')
all_uem_path  = os.path.join(UEM_DIR,  'all.uem')

with open(all_rttm_path, 'w') as outf:
    for uri in train_uris + test_uris:
        src = os.path.join(RTTM_DIR, f"{uri}.rttm")
        with open(src) as inf:
            outf.write(inf.read())

with open(all_uem_path, 'w') as outf:
    for uri in train_uris + test_uris:
        src = os.path.join(UEM_DIR, f"{uri}.uem")
        with open(src) as inf:
            outf.write(inf.read())

yml_content = f"""Databases:
  VoxConverseMini:
    train:
      - {TRAIN_AUDIO_DIR}/{{uri}}.wav
    test:
      - {TEST_AUDIO_DIR}/{{uri}}.wav

Protocols:
  VoxConverseMini:
    SpeakerDiarization:
      MiniProtocol:
        train:
          annotation: {all_rttm_path}
          annotated: {all_uem_path}
          uri: {LST_DIR}/train.lst
        development:
          annotation: {all_rttm_path}
          annotated: {all_uem_path}
          uri: {LST_DIR}/development.lst
        test:
          annotation: {all_rttm_path}
          annotated: {all_uem_path}
          uri: {LST_DIR}/test.lst
"""

with open(DB_YML_PATH, 'w') as f:
    f.write(yml_content)

print("✅ database.yml đã tạo:")
print(f"   {DB_YML_PATH}")
print()
print(yml_content)

In [ ]:
from pyannote.database import get_protocol, registry
registry.load_database(DB_YML_PATH)

# ============================================================
# Audio path helper
# ============================================================
def get_audio_path(f):
    uri = f['uri']
    if uri.startswith('train_'):
        return os.path.join(TRAIN_AUDIO_DIR, f'{uri}.wav')
    return os.path.join(TEST_AUDIO_DIR, f'{uri}.wav')

# ============================================================
# Load protocol
# ============================================================
print("\nLoading dataset protocol...")
dataset = get_protocol(
    'VoxConverseMini.SpeakerDiarization.MiniProtocol',
    {'audio': get_audio_path}
)

train_files = list(dataset.train())
dev_files   = list(dataset.development())
test_files  = list(dataset.test())

print(f"✓ Dataset loaded successfully")
print(f"  - Training files   : {len(train_files)}")
print(f"  - Development files: {len(dev_files)}")
print(f"  - Test files       : {len(test_files)}")

In [ ]:
from pyannote.audio import Model, Inference

# ============================================================
# AUTO-DETECT CHECKPOINT TỪ PART TRƯỚC
# ============================================================
PREVIOUS_PART = CURRENT_PART - 1
PREVIOUS_PART_CKPT = None

if PREVIOUS_PART >= 1:
    prev_ckpt_dir = os.path.join(BASE_DRIVE_DIR, f'checkpoints_part_{PREVIOUS_PART}')
    if os.path.isdir(prev_ckpt_dir):
        ckpt_files = [f for f in os.listdir(prev_ckpt_dir) if f.endswith('.ckpt')]
        best_ckpts = sorted([f for f in ckpt_files if f.startswith('best-model')])
        last_ckpts = [f for f in ckpt_files if f == 'last.ckpt']

        if best_ckpts:
            PREVIOUS_PART_CKPT = os.path.join(prev_ckpt_dir, best_ckpts[-1])
        elif last_ckpts:
            PREVIOUS_PART_CKPT = os.path.join(prev_ckpt_dir, 'last.ckpt')

# ============================================================
# LOAD SEGMENTATION MODEL
# ============================================================
print("Loading segmentation model architecture from HuggingFace...")
pretrained = Model.from_pretrained('pyannote/segmentation-3.0', token=HF_TOKEN)
print("✓ Architecture loaded")

if PREVIOUS_PART_CKPT is not None:
    print(f"\n🔄 Found checkpoint from Part {PREVIOUS_PART}: {os.path.basename(PREVIOUS_PART_CKPT)}")
    print("   Loading weights...")

    ckpt = torch.load(PREVIOUS_PART_CKPT, map_location=device)

    assert 'state_dict' in ckpt, (
        f"❌ Key 'state_dict' không tồn tại trong checkpoint!\n"
        f"   Các key có trong file: {list(ckpt.keys())}"
    )

    state_dict = ckpt['state_dict']
    classifier_keys = [k for k in state_dict.keys() if k.startswith('classifier')]
    for k in classifier_keys:
        del state_dict[k]
    if classifier_keys:
        print(f"  ℹ️  Bỏ qua classifier keys (size mismatch): {classifier_keys}")

    missing, unexpected = pretrained.load_state_dict(state_dict, strict=False)

    # Chỉ chấp nhận missing ở classifier (vừa bị xóa), backbone phải khớp hoàn toàn
    backbone_missing = [k for k in missing if not k.startswith('classifier')]
    assert not backbone_missing, (
        f"❌ Missing keys ở backbone: {backbone_missing}"
    )
    assert not unexpected, (
        f"❌ Unexpected keys: {unexpected}"
    )

    print(f"✓ Backbone weights từ Part {PREVIOUS_PART} loaded thành công!")
    print(f"  ℹ️  classifier layer sẽ được re-init theo dataset Part {CURRENT_PART}")
    print(f"  (checkpoint epoch: {ckpt.get('epoch', 'N/A')})")

else:
    print("\n🚀 Part 1 — dùng pretrained weights từ HuggingFace (không có checkpoint trước)")

# ============================================================
# SUMMARY
# ============================================================
print("\n" + "="*50)
print(f"  CURRENT_PART     : {CURRENT_PART}")
print(f"  Weights source   : {'Part ' + str(PREVIOUS_PART) + ' checkpoint' if PREVIOUS_PART_CKPT else 'HuggingFace pretrained'}")
if PREVIOUS_PART_CKPT:
    print(f"  Checkpoint used  : {os.path.basename(PREVIOUS_PART_CKPT)}")
print("="*50)

In [ ]:
from pyannote.audio.tasks import SpeakerDiarization as SpeakerDiarizationTask

# Reload protocol (iterator chỉ dùng 1 lần)
dataset = get_protocol(
    'VoxConverseMini.SpeakerDiarization.MiniProtocol',
    {'audio': get_audio_path}
)

print("Creating speaker diarization task...")
seg_task = SpeakerDiarizationTask(
    dataset,
    duration=10.0,
    max_speakers_per_chunk=3,
    max_speakers_per_frame=1
)

print("✓ SpeakerDiarization task created")
print(f"  - Chunk duration       : 10.0 seconds")
print(f"  - Max speakers per chunk: 3")
print(f"  - Max speakers per frame: 1")

In [ ]:
from copy import deepcopy
import lightning.pytorch as pl
from lightning.pytorch.callbacks import ModelCheckpoint

print("Creating model for fine-tuning...")
finetuned = deepcopy(pretrained)
finetuned.task = seg_task
print("✓ Model cloned and task assigned")

# Cấu hình lưu thẳng vào folder model_checkpoints của part hiện tại
checkpoint_callback = ModelCheckpoint(
    dirpath=CHECKPOINT_DIR,
    filename='best-model-epoch={epoch:02d}',
    save_top_k=1,
    monitor='loss/train',
    mode='min',
    save_last=True,
    verbose=True
)

trainer = pl.Trainer(
    devices=1,
    accelerator="gpu",
    max_epochs=50,
    default_root_dir=OUTPUT_DIR,
    callbacks=[checkpoint_callback],
    enable_checkpointing=True,
)
print(f"✓ Trainer configured with max_epochs: {trainer.max_epochs}")

print("\n" + "="*60)
print(f"STARTING FINE-TUNING FOR PART {CURRENT_PART}")
print("="*60)

trainer.fit(finetuned)

print("\n" + "="*60)
print(f"FINE-TUNING COMPLETE FOR PART {CURRENT_PART}")
print("="*60)

if os.path.exists(CHECKPOINT_DIR):
    ckpts = os.listdir(CHECKPOINT_DIR)
    print(f"\n✓ Checkpoints saved securely in: {CHECKPOINT_DIR}")
    for ckpt in ckpts:
        size_mb = os.path.getsize(os.path.join(CHECKPOINT_DIR, ckpt)) / (1024**2) if os.path.exists(os.path.join(CHECKPOINT_DIR, ckpt)) else 0
        print(f"  - {ckpt} ({size_mb:.1f} MB)")
    print(f"\n🏆 Best checkpoint path for next part reference: {checkpoint_callback.best_model_path}")